# 01 · Prepare the data

This notebook collects the audio the model learns from and checks it.

| Source | What we take from it | Download |
|---|---|---|
| Google Speech Commands v2 | `on`, `off`, 33 other words for the "unknown" class, background noise | 2.4 GB, kept on disk |
| Multilingual Spoken Words Corpus (MSWC), English | `crystal`, `bright`, `low`, more `off`, and words that sound similar to our keywords | 35 GB **streamed**, about 1 GB kept |

Both steps skip themselves when the data is already in `data/`, so the notebook is safe to re-run.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's C++ info messages

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from IPython.display import Audio, display

from fairy_kws import config, dataset, mswc
from fairy_kws.features import micro_features

print("keywords:", config.KEYWORDS)

## 1. Google Speech Commands

One-second clips of 35 words from about 2,600 speakers, one folder per word. It has `on` and `off`
but not `crystal`, `bright` or `low`.

In [ ]:
gsc_dir = dataset.download_speech_commands()

gsc_counts = pd.Series(
    {d.name: len(list(d.glob("*.wav"))) for d in sorted(gsc_dir.iterdir()) if d.is_dir()}
)
print(f"{gsc_counts.drop('_background_noise_').sum():,} clips, {len(gsc_counts) - 1} words")
gsc_counts.drop("_background_noise_").sort_values().plot.barh(
    figsize=(6, 7), title="Speech Commands clips per word"
)
plt.show()

## 2. What MSWC has for our words

MSWC cuts single words out of Mozilla Common Voice sentences. Its metadata file (103 MB) lists every
word with its clips. Words shorter than three letters are not in it, which is why `on` has to come
from Speech Commands.

In [ ]:
index = mswc.load_english_index()
print(f"{len(index):,} English words in MSWC")

candidates = [*config.MSWC_KEYWORD_CANDIDATES, "on", "brighter", "dimmer"]
pd.DataFrame(
    {"clips in MSWC": [len(index.get(word, [])) for word in candidates]}, index=candidates
).T

`crystal`, `bright` and `low` are well covered. `lamp`, the first wake word we tried, has only a
few hundred clips. `dim` has too few clips to train on, which is why `low` is the counterpart to
`bright`. `dark` is well covered too.

## 3. Fetch the MSWC clips

English audio is published as a single 35 GB archive. The next cell reads it as a stream, keeps only
the clips for the words listed in `fairy_kws/config.py` (converted to 16 kHz WAV), and stops as soon
as it has them all. Expect **40 to 90 minutes**, depending on the connection; the archive itself is never
stored. If it is interrupted, re-running continues with the clips that are still missing (the
stream starts over, finished clips are skipped).

Besides keyword clips this fetches up to 300 clips each of words such as *crystals*, *Christmas*,
*one*, *often* and *right*. They join the "unknown" class so the model has to learn the difference.

In [ ]:
mswc_counts = mswc.fetch_all()
pd.Series(mswc_counts, name="clips on disk").to_frame().T

## 4. The clip table

`build_clip_table` lists every clip with its word, source, label and split.

- **Label**: the word itself for keywords, `_unknown_` for every other word.
- **Split**: both datasets publish a train / validation / test split that keeps each speaker in a
  single set, and we use those. A model is then always tested on voices it never heard in training.

In [ ]:
clips = dataset.build_clip_table()
print(f"{len(clips):,} clips")
clips.groupby(["label", "source", "split"]).size().unstack("split")[["train", "val", "test"]]

Every keyword has more than a thousand training clips. `crystal`, `bright` and `low` come only from MSWC
and `on` only from Speech Commands, so the model could learn the recording style along with the
word. Our own recordings of all five words are the most valuable data to add later.

## 5. Listen

One random clip per keyword and source. MSWC clips are cut from running sentences, so they often
contain a bit of the neighbouring words; Speech Commands clips are isolated words.

In [ ]:
keyword_clips = clips[clips["label"] != config.UNKNOWN_LABEL]
for (label, source), group in keyword_clips.groupby(["label", "source"]):
    path = group.sample(1, random_state=3)["path"].item()
    print(f"{label}  ({source})")
    display(Audio(path))

## 6. What the model sees

The model never sees the waveform. Each clip is turned into a 49 × 40 feature map by the same audio
front end that runs on the Arduino (40 frequency bands, one column every 20 ms).

In [ ]:
fig, axes = plt.subplots(
    len(config.KEYWORDS), 4, figsize=(11, 2.1 * len(config.KEYWORDS)), constrained_layout=True
)
for row, keyword in zip(axes, config.KEYWORDS, strict=True):
    sample = keyword_clips[keyword_clips["label"] == keyword].sample(4, random_state=1)
    for ax, (_, clip) in zip(row, sample.iterrows(), strict=True):
        features = micro_features(dataset.load_clip(tf.constant(clip["path"]))).numpy()
        ax.imshow(features.T, origin="lower", aspect="auto")
        ax.set_title(f"{keyword} ({clip['source']})", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
plt.show()

Notice the difference between the two sources. Speech Commands clips are full one-second
recordings, with room noise before and after the word. MSWC clips are short cuts (often only 0.2
to 0.4 s of speech) padded to one second with **digital silence**, the flat dark areas. A real
microphone never delivers digital silence, so the training notebook mixes background noise into
every clip and also tests with noise added.

The data is ready. Continue with `02_train_keywords.ipynb`.